RAG Pipeline - Data Ingestion to Vector DB Pipeline

In [1]:
import os
from langchain_community.document_loaders import PyPDFLoader, PyMuPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from pathlib import Path

C:\Users\91730\AppData\Local\Temp\ipykernel_5936\3933654057.py:2: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import PyPDFLoader, PyMuPDFLoader
c:\Users\91730\Downloads\Rag\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
#Read all the pdf files in the directory
def process_all_pdfs(pdf_directory):
    """Process all PDF files in the specified directory and return a list of documents."""
    all_documents = []
    pdf_dir = Path(pdf_directory)
    
    #find all pdf files in the directory
    pdf_files = list(pdf_dir.glob("**/*.pdf"))
    print(f"Found {len(pdf_files)} PDF files in the to process")
    
    for pdf_file in pdf_files:
        print(f"\nProcessing: {pdf_file.name}...")
        try:
            loader = PyPDFLoader(str(pdf_file))
            documents = loader.load()
            
            # Add source information to metadata
            for doc in documents:
                doc.metadata["source_file"] = pdf_file.name
                doc.metadata["file_path"] = "pdf"
                
            all_documents.extend(documents)
            print(f" Loaded {len(documents)} pages")
            
        except Exception as e:
            print(f" Error: {e}")

    print(f"\nTotal documents loaded: {len(all_documents)}")
    return all_documents

#process all pdf files in the directory
all_pdf_documents = process_all_pdfs("../data")


Found 1 PDF files in the to process

Processing: NLP.pdf...
 Loaded 31 pages

Total documents loaded: 31


In [3]:
all_pdf_documents


[Document(metadata={'producer': 'www.ilovepdf.com', 'creator': 'Microsoft® PowerPoint® 2016', 'creationdate': '2023-11-02T04:09:03+00:00', 'moddate': '2023-11-02T04:09:03+00:00', 'source': '..\\data\\pdf\\NLP.pdf', 'total_pages': 31, 'page': 0, 'page_label': '1', 'source_file': 'NLP.pdf', 'file_path': 'pdf'}, page_content='NATURAL LANGUAGE \nPROCESSING (NLP)'),
 Document(metadata={'producer': 'www.ilovepdf.com', 'creator': 'Microsoft® PowerPoint® 2016', 'creationdate': '2023-11-02T04:09:03+00:00', 'moddate': '2023-11-02T04:09:03+00:00', 'source': '..\\data\\pdf\\NLP.pdf', 'total_pages': 31, 'page': 1, 'page_label': '2', 'source_file': 'NLP.pdf', 'file_path': 'pdf'}, page_content='Natural Language Processing\n• Gives computers the ability to understand text and spoken words \nlike human do.\n• It combines statistical, machine learning and deep learning models \nto achieve this.\n• Human text and voice data is broken down using various tasks as \nfollows:\nSpeech recognition\n• Also call

In [4]:
##Text splitting
def split_documents(documents, chunk_size=1000, chunk_overlap=200):
    """Split documents into smaller chunks using RecursiveCharacterTextSplitter."""
    text_splitter = RecursiveCharacterTextSplitter(
        chunk_size=chunk_size,
        chunk_overlap=chunk_overlap,
        length_function=len,
        separators=["\n\n", "\n", " ", ""]
    )
    split_docs=text_splitter.split_documents(documents)
    print(f"split{len(documents)} documents into {len(split_docs)} chunks")
    
    #show example of the first chunk
    if split_docs:
        print(f"\nExample of the first chunk:")
        print(f"Content: {split_docs[0].page_content[:200]}...") 
        print(f"Metadata:{split_docs[0].metadata}")
    return split_docs

In [5]:
chunks=split_documents(all_pdf_documents)

split31 documents into 29 chunks

Example of the first chunk:
Content: NATURAL LANGUAGE 
PROCESSING (NLP)...
Metadata:{'producer': 'www.ilovepdf.com', 'creator': 'Microsoft® PowerPoint® 2016', 'creationdate': '2023-11-02T04:09:03+00:00', 'moddate': '2023-11-02T04:09:03+00:00', 'source': '..\\data\\pdf\\NLP.pdf', 'total_pages': 31, 'page': 0, 'page_label': '1', 'source_file': 'NLP.pdf', 'file_path': 'pdf'}


Embedding and Vector storeDB


In [6]:
import numpy as np
from sentence_transformers import SentenceTransformer
import chromadb
from chromadb.config import Settings
import uuid
from typing import List, Dict, Any,Tuple
from sklearn.metrics.pairwise import cosine_similarity

In [7]:
class EmbeddingManger:
    """Handler for managing embeddings."""

    def __init__(self, model_name: str = 'all-MiniLM-L6-v2'):
        """
        Initialize the EmbeddingManager with a specified model name.

        Args:
            model_name: HuggingFace model name for sentence embeddings
        """
        self.model_name = model_name
        self.model = None
        self._load_model()

    def _load_model(self):
        """Load the sentence transformer model."""
        try:
            print(f"Loading embedding model: {self.model_name}")
            self.model = SentenceTransformer(self.model_name)
            print(f"Model loaded successfully. Embedding dimension: {self.model.get_sentence_embedding_dimension()}")
        except Exception as e:
            print(f"Error loading model {self.model_name}: {e}")
            return None

    def generate_embeddings(self, texts: list) -> np.ndarray:
        """
        Generate embeddings for a list of texts.

        Args:
            texts: List of strings to embed

        Returns:
            A NumPy array of embeddings
        """
        if self.model is None:
            raise RuntimeError("Embedding model is not loaded")
        return np.array(self.model.encode(texts, show_progress_bar=False, convert_to_numpy=True))

###initialize the embedding manager
embedding_manager = EmbeddingManger()
embedding_manager


Loading embedding model: all-MiniLM-L6-v2


Loading weights: 100%|██████████| 103/103 [00:00<00:00, 1596.89it/s]


Model loaded successfully. Embedding dimension: 384


C:\Users\91730\AppData\Local\Temp\ipykernel_5936\647314322.py:20: FutureWarning: The `get_sentence_embedding_dimension` method has been renamed to `get_embedding_dimension`.
  print(f"Model loaded successfully. Embedding dimension: {self.model.get_sentence_embedding_dimension()}")


Vector Store

In [8]:
import os

class VectorStore:
    """Handler document embedding in a ChromaDB vector store."""

    def __init__(self, collection_name: str = "pdf_collection", persist_directory: str = "../data/vector_store"):
        """
        Initialize the VectorStore with a specified collection name.

        Args:
            collection_name: Name of the ChromaDB collection
            persist_directory: Path to persist the ChromaDB files
        """
        self.collection_name = collection_name
        self.persist_directory = persist_directory
        self.client = None
        self.collection = None
        self._initialize_store()

    def _initialize_store(self):
        """Initialize the ChromaDB client and collection."""
        try:
            os.makedirs(self.persist_directory, exist_ok=True)
            self.client = chromadb.PersistentClient(path=self.persist_directory)
            self.collection = self.client.get_or_create_collection(
                name=self.collection_name,
                metadata={"description": "PDF document embeddings for RAG"}
            )
            print(f"Vector store initialized with collection: {self.collection_name}")
            print(f"Existing documents in collection: {self.collection.count()}")
        except Exception as e:
            print(f"Error initializing vector store: {e}")
            raise

    def add_documents(self, documents: List[Any], embeddings: np.ndarray):
        """
        Add documents to the vector store.

        Args:
            documents: List of Langchain documents
            embeddings: Corresponding embeddings for the documents
        """
        if len(documents) != len(embeddings):
            raise ValueError("Number of documents must match number of embeddings.")

        print(f"Adding {len(documents)} documents to the vector store...")

        ids = []
        metadatas = []
        documents_text = []
        embeddings_list = []

        for i, (doc, embedding) in enumerate(zip(documents, embeddings)):
            doc_id = f"{uuid.uuid4().hex[:8]}_{i}"
            ids.append(doc_id)

            metadata = dict(doc.metadata)
            metadata["doc_index"] = i
            metadata["content_length"] = len(doc.page_content)
            metadatas.append(metadata)

            documents_text.append(doc.page_content)
            embeddings_list.append(embedding.tolist())

        try:
            self.collection.add(
                ids=ids,
                embeddings=embeddings_list,
                metadatas=metadatas,
                documents=documents_text
            )
            print(f"Successfully added {len(documents)} documents to vector store")
            print(f"Total documents in collection: {self.collection.count()}")
        except Exception as e:
            print(f"Error adding documents to vector store: {e}")
            raise

vectorstore = VectorStore()
vectorstore
    

Vector store initialized with collection: pdf_collection
Existing documents in collection: 116


In [9]:
all_pdf_documents

[Document(metadata={'producer': 'www.ilovepdf.com', 'creator': 'Microsoft® PowerPoint® 2016', 'creationdate': '2023-11-02T04:09:03+00:00', 'moddate': '2023-11-02T04:09:03+00:00', 'source': '..\\data\\pdf\\NLP.pdf', 'total_pages': 31, 'page': 0, 'page_label': '1', 'source_file': 'NLP.pdf', 'file_path': 'pdf'}, page_content='NATURAL LANGUAGE \nPROCESSING (NLP)'),
 Document(metadata={'producer': 'www.ilovepdf.com', 'creator': 'Microsoft® PowerPoint® 2016', 'creationdate': '2023-11-02T04:09:03+00:00', 'moddate': '2023-11-02T04:09:03+00:00', 'source': '..\\data\\pdf\\NLP.pdf', 'total_pages': 31, 'page': 1, 'page_label': '2', 'source_file': 'NLP.pdf', 'file_path': 'pdf'}, page_content='Natural Language Processing\n• Gives computers the ability to understand text and spoken words \nlike human do.\n• It combines statistical, machine learning and deep learning models \nto achieve this.\n• Human text and voice data is broken down using various tasks as \nfollows:\nSpeech recognition\n• Also call

Convert the text to embeddings

In [10]:
texts=[doc.page_content for doc in chunks]
texts

['NATURAL LANGUAGE \nPROCESSING (NLP)',
 'Natural Language Processing\n• Gives computers the ability to understand text and spoken words \nlike human do.\n• It combines statistical, machine learning and deep learning models \nto achieve this.\n• Human text and voice data is broken down using various tasks as \nfollows:\nSpeech recognition\n• Also called speech-to-text.\n• It is challenging as people may speak in different accents and \nintonations and often using incorrect grammar.\nPart of speech tagging\n• Also called grammatical tagging.\n• It determines the part of speech (i.e., verb , noun etc.) of a particular \nword in a sentence.',
 'Word of sense disambiguation\n• Selection of the meaning of a particular word with multiple \nmeanings that makes the most sense in the given context.\nNamed entity recognition (NER)\n• Identifies words or phrases as useful entities.\n• It identifies ‘John’ as a man’s name and ‘London’ as a place.\nCo-reference resolution\n• Identifies if and when 

In [11]:
#Generate the embeddingd

embeddings = embedding_manager.generate_embeddings(texts)

#store in the vector database
vectorstore.add_documents(chunks, embeddings)

Adding 29 documents to the vector store...
Successfully added 29 documents to vector store
Total documents in collection: 145


Retriever Pipeline From VectorStore

In [12]:
class RAGRetriever:
    """Handles query based retrieval from the vector store."""

    def __init__(self, vector_store: VectorStore, embedding_manager: EmbeddingManger):
        """Initialize the retriever.

        Args:
            vector_store: vector store containing document embeddings
            embedding_manager: Manager for generating query embeddings
        """
        self.vector_store = vector_store
        self.embedding_manager = embedding_manager

    def retrieve(self, query: str, top_k: int = 5, score_threshold: float = 0.0) -> List[Dict[str, Any]]:
        """Retrieve relevant documents for a query.

        Args:
            query: The search query
            top_k: Number of top results to return
            score_threshold: Minimum similarity score threshold

        Returns:
            List of dictionaries containing retrieved documents and metadata
        """
        print(f"Retrieving documents for query: {query}")
        print(f"Top K: {top_k}, Score threshold: {score_threshold}")

        try:
            query_embedding = self.embedding_manager.generate_embeddings([query])[0]
            results = self.vector_store.collection.query(
                query_embeddings=[query_embedding.tolist()],
                n_results=top_k
            )

            retrieved_docs = []
            if results.get('documents') and results['documents'][0]:
                documents = results['documents'][0]
                metadatas = results.get('metadatas', [[]])[0]
                distances = results.get('distances', [[]])[0]
                ids = results.get('ids', [[]])[0]

                for i, (doc_id, document, metadata, distance) in enumerate(zip(ids, documents, metadatas, distances)):
                    similarity_score = 1 - distance
                    if similarity_score >= score_threshold:
                        retrieved_docs.append({
                            'id': doc_id,
                            'content': document,
                            'metadata': metadata,
                            'similarity_score': similarity_score,
                            'distance': distance,
                            'rank': i + 1
                        })

            if retrieved_docs:
                print(f"Retrieved {len(retrieved_docs)} documents (after filtering)")
            else:
                print("No documents found")

            return retrieved_docs

        except Exception as e:
            print(f"Error during retrieval: {e}")
            return []


rag_retriever = RAGRetriever(vectorstore, embedding_manager)
        

In [13]:
rag_retriever

In [14]:
rag_retriever.retrieve("EXplain Limitations of ngram modeling")

Retrieving documents for query: EXplain Limitations of ngram modeling
Top K: 5, Score threshold: 0.0
Retrieved 5 documents (after filtering)


[{'id': '7f781204_15',
  'content': 'Limitations of ngram modeling\n• The higher the n, the better the model, but higher will be the \ncomputational cost.\n• It is based on probability of words co-occurring. If the words are not \npresent in the training corpus, it will give zero probability to those \nwords. If such ngrams occur in testing set, the overall probability of \nthe test sentence becomes 0.',
  'metadata': {'creationdate': '2023-11-02T04:09:03+00:00',
   'doc_index': 15,
   'moddate': '2023-11-02T04:09:03+00:00',
   'creator': 'Microsoft® PowerPoint® 2016',
   'source_file': 'NLP.pdf',
   'content_length': 362,
   'total_pages': 31,
   'file_path': 'pdf',
   'page_label': '17',
   'page': 16,
   'producer': 'www.ilovepdf.com',
   'source': '..\\data\\pdf\\NLP.pdf'},
  'similarity_score': 0.4262409806251526,
  'distance': 0.5737590193748474,
  'rank': 1},
 {'id': '2e940f9d_15',
  'content': 'Limitations of ngram modeling\n• The higher the n, the better the model, but higher 

Integration Vectordb Context pipeline with LLM output

In [22]:
##simple RAG pipeline with Groq LLM
from langchain_google_genai import ChatGoogleGenerativeAI
import os
from dotenv import load_dotenv
load_dotenv()

#initialize the Google Vertex AI model (set GOOGLE_APPLICATION_CREDENTIALS in environment)
llm = ChatGoogleGenerativeAI(model="gemini-2.5-flash", temperature=0.1)

#2.Simple RAG function :retrieve context +generate response
def rag_simple(query, retriever, llm, top_k=3):
    ##retriever the context
    results = retriever.retrieve(query, top_k=top_k)
    context="\n\n".join([doc['content']for doc in results]) if results else""
    if not context:
        return "No relevant context found to answer the question."
    
    #generate the answer using Groq LLM
    prompt=f"""Use the following context to answer the questio concisely.
        Context:
        {context}
        Question:{query}
    
        Answer:"""
        
    response=llm.invoke([prompt.format(context=context,query=query)])
    return response.content

In [23]:
answer=rag_simple("Explain Limitations of ngram modeling",rag_retriever,llm)
print(answer)

Retrieving documents for query: Explain Limitations of ngram modeling
Top K: 3, Score threshold: 0.0
Retrieved 3 documents (after filtering)
Ngram modeling has two main limitations:
1.  Higher 'n' (order of the ngram) improves the model but significantly increases computational cost.
2.  It assigns zero probability to ngrams not present in the training corpus, which can lead to an overall probability of zero for test sentences containing such unseen ngrams.


Enhanced RAG Pipeline Features


In [26]:
# --- Enhanced RAG Pipeline Features ---
def rag_advanced(query, retriever, llm, top_k=5, min_score=0.2, return_context=False):
    """
    RAG pipeline with extra features:
    - Returns answer, sources, confidence score, and optionally full context.
    """
    results = retriever.retrieve(query, top_k=top_k, score_threshold=min_score)
    if not results:
        return {'answer': 'No relevant context found.', 'sources': [], 'confidence': 0.0, 'context': ''}
    
    # Prepare context and sources
    context = "\n\n".join([doc['content'] for doc in results])
    sources = [{
        'source': doc['metadata'].get('source_file', doc['metadata'].get('source', 'unknown')),
        'page': doc['metadata'].get('page', 'unknown'),
        'score': doc['similarity_score'],
        'preview': doc['content'][:300] + '...'
    } for doc in results]
    confidence = max([doc['similarity_score'] for doc in results])
    
    # Generate answer
    prompt = f"""Use the following context to answer the question concisely.\nContext:\n{context}\n\nQuestion: {query}\n\nAnswer:"""
    response = llm.invoke([prompt.format(context=context, query=query)])
    
    output = {
        'answer': response.content,
        'sources': sources,
        'confidence': confidence
    }
    if return_context:
        output['context'] = context
    return output

# Example usage:
result = rag_advanced("Explain N-gram language model unigram,bigram, trigram", rag_retriever, llm, top_k=3, min_score=0.1, return_context=True)
print("Answer:", result['answer'])
print("Sources:", result['sources'])
print("Confidence:", result['confidence'])
print("Context Preview:", result['context'][:300])

Retrieving documents for query: Explain N-gram language model unigram,bigram, trigram
Top K: 3, Score threshold: 0.1
Retrieved 3 documents (after filtering)
Answer: An N-gram language model is a statistical model where an N-gram is a sequence of N tokens.
*   **Unigram (1-gram):** A one-word sequence (e.g., 'The', 'ability').
*   **Bigram (2-gram):** A two-word sequence (e.g., ‘The ability’, ‘ability to’).
*   **Trigram (3-gram):** A three-word sequence (e.g., ‘The ability to’, ‘ability to model’).
Sources: [{'source': 'NLP.pdf', 'page': 13, 'score': 0.6791325211524963, 'preview': 'N-gram language model (unigram, \nbigram, trigram)—Statistical model\n• N-gram is a sequence of N tokens.\nEg: “The ability to model the rules of a language as a probability \ngives great power for NLP related tasks ”\n• A 1-gram (unigram) is a one-word sequence. \nHere, the unigrams would be ‘The’, ‘abil...'}, {'source': 'NLP.pdf', 'page': 13, 'score': 0.6791325211524963, 'preview': 'N-gram language model (

In [28]:
# --- Advanced RAG Pipeline: Streaming, Citations, History, Summarization ---
from typing import List, Dict, Any
import time

class AdvancedRAGPipeline:
    def __init__(self, retriever, llm):
        self.retriever = retriever
        self.llm = llm
        self.history = []  # Store query history

    def query(self, question: str, top_k: int = 5, min_score: float = 0.2, stream: bool = False, summarize: bool = False) -> Dict[str, Any]:
        # Retrieve relevant documents
        results = self.retriever.retrieve(question, top_k=top_k, score_threshold=min_score)
        if not results:
            answer = "No relevant context found."
            sources = []
            context = ""
        else:
            context = "\n\n".join([doc['content'] for doc in results])
            sources = [{
                'source': doc['metadata'].get('source_file', doc['metadata'].get('source', 'unknown')),
                'page': doc['metadata'].get('page', 'unknown'),
                'score': doc['similarity_score'],
                'preview': doc['content'][:120] + '...'
            } for doc in results]
            # Streaming answer simulation
            prompt = f"""Use the following context to answer the question concisely.\nContext:\n{context}\n\nQuestion: {question}\n\nAnswer:"""
            if stream:
                print("Streaming answer:")
                for i in range(0, len(prompt), 80):
                    print(prompt[i:i+80], end='', flush=True)
                    time.sleep(0.05)
                print()
            response = self.llm.invoke([prompt.format(context=context, question=question)])
            answer = response.content

        # Add citations to answer
        citations = [f"[{i+1}] {src['source']} (page {src['page']})" for i, src in enumerate(sources)]
        answer_with_citations = answer + "\n\nCitations:\n" + "\n".join(citations) if citations else answer

        # Optionally summarize answer
        summary = None
        if summarize and answer:
            summary_prompt = f"Summarize the following answer in 2 sentences:\n{answer}"
            summary_resp = self.llm.invoke([summary_prompt])
            summary = summary_resp.content

        # Store query history
        self.history.append({
            'question': question,
            'answer': answer,
            'sources': sources,
            'summary': summary
        })

        return {
            'question': question,
            'answer': answer_with_citations,
            'sources': sources,
            'summary': summary,
            'history': self.history
        }

# Example usage:
adv_rag = AdvancedRAGPipeline(rag_retriever, llm)
result = adv_rag.query("Explain N-gram language model unigram,bigram, trigram", top_k=3, min_score=0.1, stream=True, summarize=True)
print("\nFinal Answer:", result['answer'])
print("Summary:", result['summary'])
print("History:", result['history'][-1])

Retrieving documents for query: Explain N-gram language model unigram,bigram, trigram
Top K: 3, Score threshold: 0.1
Retrieved 3 documents (after filtering)
Streaming answer:
Use the following context to answer the question concisely.
Context:
N-gram language model (unigram, 
bigram, trigram)—Statistical model
• N-gram is a sequence of N tokens.
Eg: “The ability to model the rules of a language as a probability 
gives great power for NLP related tasks ”
• A 1-gram (unigram) is a one-word sequence. 
Here, the unigrams would be ‘The’, ‘ability’, ‘to’, ‘model’, etc.
• A 2-gram (bigram) is a two-word sequence of words.
‘The ability’, ‘ability to’, ‘to model’, ‘model the’ etc.
• A 3-gram (trigram) is a three-word sequence of words.
‘The ability to’, ‘ability to model’ etc.

N-gram language model (unigram, 
bigram, trigram)—Statistical model
• N-gram is a sequence of N tokens.
Eg: “The ability to model the rules of a language as a probability 
gives great power for NLP related tasks ”
• A 1-